# 🎬 《霧港館的第六個人》美術館第一人稱過場影片生成器（Colab A100 GPU）

本 Notebook 專為 **Google Colab A100 GPU** 設計，讀取專案 `assets/scene_*.jpg` 六大美術館場景插圖，使用開源圖生影片（Image-to-Video）模型生成 **2～3 秒第一人稱走路／開門過場 `.mp4` 影片**，並自動轉碼為網頁相容格式（H.264 `yuv420p` + `faststart`）存入你的 Google Drive：

- **輸出路徑**：`MyDrive/Final-Project-Videos/`
- **支援開源模型（二選一，免付費 API Key）**：
  1. **`LTX-Video`（推薦預設）**：`Lightricks/LTX-Video-0.9.5`，A100 生成一支 2.5 秒影片僅約 20～35 秒。
  2. **`Wan2.1-I2V-14B-480P`**：`Wan-AI/Wan2.1-I2V-14B-480P-Diffusers`，電影級運鏡質感，A100 40GB/80GB 皆可執行。
- **與 Vercel 網頁版無縫整合**：生成後的 `scene_museum.mp4`、`scene_security.mp4`、`scene_basement.mp4`、`scene_restoration.mp4`、`scene_loading_dock.mp4`、`scene_curator_office.mp4` 只要放入專案 `assets/videos/` 並推送到 GitHub `main`，網頁版就會自動優先播放真實影片過場！

> **執行前確認**：上方選單「執行階段 → 變更執行階段類型 → **A100 GPU**」，接著點選「執行階段 → **全部執行**」。

In [ ]:
#@title 1. 檢查 A100 GPU 環境並掛載 Google Drive
import os
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), "❌ 未偵測到 GPU！請到「執行階段 → 變更執行階段類型」選擇 A100 GPU。"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"✅ 已連線 GPU：{gpu_name}（VRAM: {vram_gb:.1f} GB）")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

drive.mount("/content/drive", force_remount=False)
OUTPUT_DIR = Path("/content/drive/MyDrive/Final-Project-Videos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Google Drive 輸出資料夾已就緒：{OUTPUT_DIR}")

In [ ]:
#@title 2. 安裝 Diffusers、Accelerate 與影片輸出相依套件
!pip install -q --upgrade "diffusers>=0.32.2" "transformers>=4.49.0" "accelerate>=1.2.0" sentencepiece ftfy imageio imageio-ffmpeg einops av
!ffmpeg -version | head -n 2
import diffusers, transformers
print(f"✅ diffusers {diffusers.__version__} | transformers {transformers.__version__}")

In [ ]:
#@title 3. 複製 erickuo0707/Final-Project 並讀取 assets/scene_*.jpg
import subprocess
from pathlib import Path
from PIL import Image

REPO_URL = "https://github.com/erickuo0707/Final-Project.git"
REPO_DIR = Path("/content/Final-Project")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)

ASSETS_DIR = REPO_DIR / "assets"
scene_files = sorted(ASSETS_DIR.glob("scene_*.jpg"))
print(f"🖼️ 找到 {len(scene_files)} 張場景圖：")
for p in scene_files:
    with Image.open(p) as im:
        print(f"  - {p.name:28s} ({im.width}x{im.height})")

In [ ]:
#@title 4. 影片生成參數設定與《紙嫁衣》懸疑美術館運鏡 Prompt
MODEL_CHOICE = "LTX-Video" #@param ["LTX-Video", "Wan2.1-I2V-14B-480P"]
DURATION_SECONDS = 2.5 #@param {type:"slider", min:2.0, max:3.0, step:0.5}
INFERENCE_STEPS = 32 #@param {type:"slider", min:20, max:50, step:2}
INCLUDE_COURTROOM = False #@param {type:"boolean"}
OVERWRITE_EXISTING = False #@param {type:"boolean"}
SEED = 1842 #@param {type:"integer"}

NEGATIVE_PROMPT = (
    "worst quality, blurry, jittery, distorted architecture, morphing walls, "
    "text watermark, sudden cuts, bright daylight, cartoonish bounce, extra people appearing"
)

# 檔名對應 web/app.js 的 SCENE_VIDEO_STEMS，放入 assets/videos/ 即可自動優先播放
SCENE_SPECS = [
    {
        "stem": "scene_museum",
        "label": "2F B 展廳（《海霧港灣》主牆）",
        "prompt": (
            "First-person POV slowly walking forward into a dimly lit quiet art museum gallery at night, "
            "subtle natural footsteps camera sway, warm golden overhead spotlight casting soft shimmer "
            "on the ornate framed seascape painting, floating dust motes in the light beam, "
            "cinematic atmospheric suspense mystery style, smooth steady dolly-in, high detail."
        ),
    },
    {
        "stem": "scene_security",
        "label": "1F 保全中控區",
        "prompt": (
            "First-person POV stepping through an arched stone doorway into a dark museum security control room at night, "
            "gentle walking head-bob motion, faint cyan glow flickering softly on dual CCTV monitors, "
            "amber and red alarm panel LEDs blinking gently on the wall, dust particles drifting in moody shadows, "
            "atmospheric mystery thriller cinematography."
        ),
    },
    {
        "stem": "scene_basement",
        "label": "B1 地下器材室",
        "prompt": (
            "First-person POV pushing past a heavy basement door and walking slowly toward a dimly lit janitor desk, "
            "subtle walking camera movement, warm tungsten desk lamp illuminating an open logbook on the wooden table, "
            "faint floating dust motes in damp underground air, eerie quiet nocturnal museum basement atmosphere."
        ),
    },
    {
        "stem": "scene_restoration",
        "label": "B1 修復室 R-07",
        "prompt": (
            "First-person POV walking slowly into a secret museum art restoration room at night, "
            "gentle stepping dolly-in toward the central wooden easel, glowing violet UV tube lamp humming softly "
            "over the seascape canvas, archival UV film lightbox glowing warmly on the right wall, "
            "drifting dust specks, moody mystery thriller lighting."
        ),
    },
    {
        "stem": "scene_loading_dock",
        "label": "1F 西側裝卸區",
        "prompt": (
            "First-person POV stepping into a cavernous museum west loading dock at night, "
            "subtle walking gait forward toward the work table and wooden climate-controlled art crate on a cart, "
            "overhead industrial pendant lamp swaying slightly, cool harbor night mist drifting near the steel shutter, "
            "cinematic suspense mystery atmosphere."
        ),
    },
    {
        "stem": "scene_curator_office",
        "label": "2F 策展人辦公室",
        "prompt": (
            "First-person POV opening a wooden door and stepping smoothly into a refined museum curator office at night, "
            "gentle walking camera push toward the mahogany desk with an open catalog under a green banker lamp "
            "and an ornate brass wooden lockbox, harbor fog drifting slowly outside the arched window, "
            "atmospheric nocturnal suspense cinematography."
        ),
    },
]

if INCLUDE_COURTROOM:
    SCENE_SPECS.append(
        {
            "stem": "scene_courtroom",
            "label": "市立霧港地方法院（選用）",
            "prompt": (
                "First-person POV walking steadily down the center aisle of a solemn wooden courtroom, "
                "warm dramatic ceiling spotlights, floating dust motes in shafts of light, "
                "cinematic legal mystery atmosphere, smooth camera push."
            ),
        }
    )

print(f"🎬 已設定模型：{MODEL_CHOICE}｜目標長度：{DURATION_SECONDS}s｜預計處理 {len(SCENE_SPECS)} 支場景影片")

In [ ]:
#@title 5. 載入開源圖生影片管線（LTX-Video / Wan2.1-I2V）
import gc
import torch

pipe = None
gc.collect()
torch.cuda.empty_cache()

if MODEL_CHOICE == "LTX-Video":
    from diffusers import LTXImageToVideoPipeline

    LTX_MODEL_ID = "Lightricks/LTX-Video-0.9.5"
    try:
        print(f"⬇️ 正在載入 {LTX_MODEL_ID} ...")
        pipe = LTXImageToVideoPipeline.from_pretrained(LTX_MODEL_ID, torch_dtype=torch.bfloat16)
    except Exception as exc:
        print(f"⚠️ 無法載入 {LTX_MODEL_ID}（{exc}），改用 Lightricks/LTX-Video ...")
        pipe = LTXImageToVideoPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.bfloat16)

    pipe.to("cuda")
    if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_tiling"):
        pipe.vae.enable_tiling()
    VIDEO_FPS = 24
    # LTX-Video 規定 num_frames = 8 * k + 1，寬高為 32 倍數（配合原圖 16:9 比例）
    TARGET_W, TARGET_H = 768, 448
    k = max(4, round((DURATION_SECONDS * VIDEO_FPS - 1) / 8))
    NUM_FRAMES = 8 * k + 1

elif MODEL_CHOICE == "Wan2.1-I2V-14B-480P":
    from diffusers import AutoencoderKLWan, WanImageToVideoPipeline

    WAN_MODEL_ID = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"
    print(f"⬇️ 正在載入 {WAN_MODEL_ID}（約 28GB，首次下載需數分鐘）...")
    vae = AutoencoderKLWan.from_pretrained(WAN_MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
    pipe = WanImageToVideoPipeline.from_pretrained(WAN_MODEL_ID, vae=vae, torch_dtype=torch.bfloat16)
    pipe.enable_model_cpu_offload()
    VIDEO_FPS = 16
    # Wan2.1 規定 num_frames = 4 * k + 1，480P 寬螢幕為 832x480
    TARGET_W, TARGET_H = 832, 480
    k = max(8, round((DURATION_SECONDS * VIDEO_FPS - 1) / 4))
    NUM_FRAMES = 4 * k + 1
else:
    raise ValueError(f"Unsupported MODEL_CHOICE: {MODEL_CHOICE}")

actual_sec = NUM_FRAMES / VIDEO_FPS
print(f"✅ 模型就緒！解析度：{TARGET_W}x{TARGET_H}｜幀數：{NUM_FRAMES} 幀 @ {VIDEO_FPS}fps（約 {actual_sec:.2f} 秒）")

In [ ]:
#@title 6. 批次生成 2～3 秒過場影片並自動轉碼存至 Google Drive
import time
import subprocess
from pathlib import Path
from PIL import Image, ImageOps
from diffusers.utils import export_to_video

TEMP_DIR = Path("/content/tmp_videos")
TEMP_DIR.mkdir(parents=True, exist_ok=True)


def prepare_image(img_path: Path, width: int, height: int) -> Image.Image:
    im = Image.open(img_path).convert("RGB")
    return ImageOps.fit(im, (width, height), method=Image.Resampling.LANCZOS)


def transcode_web_mp4(raw_mp4: Path, out_mp4: Path):
    # 轉成全瀏覽器（Chrome/Safari/iOS/Edge）秒播的 H.264 yuv420p + faststart
    cmd = [
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", str(raw_mp4),
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-profile:v", "main",
        "-crf", "21",
        "-movflags", "+faststart",
        "-an",
        str(out_mp4),
    ]
    subprocess.run(cmd, check=True)


generated_paths = []
for idx, spec in enumerate(SCENE_SPECS, start=1):
    stem = spec["stem"]
    src_img = ASSETS_DIR / f"{stem}.jpg"
    final_mp4 = OUTPUT_DIR / f"{stem}.mp4"

    if not src_img.exists():
        print(f"⚠️ 找不到來源圖片：{src_img}，略過。")
        continue
    if final_mp4.exists() and not OVERWRITE_EXISTING:
        size_kb = final_mp4.stat().st_size / 1024
        print(f"⏭️ [{idx}/{len(SCENE_SPECS)}] 已存在於 Drive：{final_mp4.name}（{size_kb:.0f} KB），略過。")
        generated_paths.append(final_mp4)
        continue

    print(f"🎬 [{idx}/{len(SCENE_SPECS)}] 開始生成 {spec['label']}（{stem}.mp4）...")
    t0 = time.time()
    init_image = prepare_image(src_img, TARGET_W, TARGET_H)
    generator = torch.Generator(device="cuda").manual_seed(SEED + idx)

    if MODEL_CHOICE == "LTX-Video":
        result = pipe(
            image=init_image,
            prompt=spec["prompt"],
            negative_prompt=NEGATIVE_PROMPT,
            width=TARGET_W,
            height=TARGET_H,
            num_frames=NUM_FRAMES,
            num_inference_steps=INFERENCE_STEPS,
            guidance_scale=3.5,
            generator=generator,
        )
    else:
        result = pipe(
            image=init_image,
            prompt=spec["prompt"],
            negative_prompt=NEGATIVE_PROMPT,
            width=TARGET_W,
            height=TARGET_H,
            num_frames=NUM_FRAMES,
            num_inference_steps=INFERENCE_STEPS,
            guidance_scale=5.0,
            generator=generator,
        )

    frames = result.frames[0]
    raw_mp4 = TEMP_DIR / f"{stem}_raw.mp4"
    export_to_video(frames, str(raw_mp4), fps=VIDEO_FPS)
    transcode_web_mp4(raw_mp4, final_mp4)

    elapsed = time.time() - t0
    size_kb = final_mp4.stat().st_size / 1024
    print(f"   ✅ 完成！耗時 {elapsed:.1f}s｜大小 {size_kb:.0f} KB｜已存至 {final_mp4}")
    generated_paths.append(final_mp4)
    torch.cuda.empty_cache()

print(f"\n🎉 全部處理完成！共 {len(generated_paths)} 支影片已保存在：{OUTPUT_DIR}")

In [ ]:
#@title 7. 在 Colab 內即時預覽 Google Drive 內的過場影片
import base64
from IPython.display import HTML, display

mp4_files = sorted(OUTPUT_DIR.glob("*.mp4"))
if not mp4_files:
    print("尚未找到任何 .mp4 影片，請先執行上一個生成儲存格。")
else:
    cards = []
    for mp4 in mp4_files:
        b64 = base64.b64encode(mp4.read_bytes()).decode("ascii")
        size_kb = mp4.stat().st_size / 1024
        cards.append(
            f'''<div style="width:420px;background:#141820;border:1px solid #8a6e40;border-radius:12px;padding:10px;color:#f3e3be;">
              <div style="font-weight:700;margin-bottom:6px;">🎬 {mp4.name} ({size_kb:.0f} KB)</div>
              <video controls autoplay loop muted playsinline style="width:100%;border-radius:8px;" src="data:video/mp4;base64,{b64}"></video>
            </div>'''
        )
    display(HTML(f'<div style="display:flex;flex-wrap:wrap;gap:14px;">{"".join(cards)}</div>'))

## 🚀 如何啟用到 Vercel 網頁版？

1. 開啟 Google Drive 的 **`MyDrive/Final-Project-Videos/`**，下載剛剛生成的 `scene_*.mp4` 影片。
2. 將這些 `.mp4` 檔案複製到 GitHub 專案的 **`assets/videos/`** 資料夾（例如 `assets/videos/scene_museum.mp4`、`assets/videos/scene_security.mp4`…）。
3. 提交並推送到 `main` 分支：
   ```bash
   git add assets/videos/*.mp4
   git commit -m "feat: add AI-generated first-person museum transition videos"
   git push origin main
   ```
4. Vercel 自動重新部署後，玩家點擊門口／樓梯熱區或切換場景時，`web/app.js` 會**自動偵測並優先播放 `assets/videos/` 裡的 `.mp4` 過場影片**（若未放入影片則自動播網頁第一人稱走路動畫）。
   - 💡 進階客製：若想為特定兩房間之間製作專屬路線影片，也可命名為 `assets/videos/walk_<出發房間id>_to_<目標房間id>.mp4`（例如 `walk_gallery_to_security.mp4`），網頁版會以最高優先順序播放！